# Deep Learning for German Credit Data (PyTorch)
This notebook implements a simple but highly regularized Multi-Layer Perceptron (MLP) for predicting credit default. 

**Why is this difficult?**
Neural Networks typically require tens of thousands of rows of data to generalize. This dataset has exactly 1,000 rows. If we train a standard Neural Network, it will memorize the training set in 5 epochs and fail completely on the test set.

**The Solution:**
1. **Aggressive Regularization:** We use heavy `Dropout` layers and `BatchNorm` to penalize the network and prevent it from memorizing specific rows.
2. **Asymmetric Loss Function:** We do not use SMOTE to handle the class imbalance. Instead, we use PyTorch's `BCEWithLogitsLoss(pos_weight=...)`, which mathematically penalizes the network heavily if it misses a "bad" loan during backpropagation.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, classification_report, precision_score, recall_score, precision_recall_curve
import warnings
warnings.filterwarnings('ignore')

# Set device to GPU if available on Kaggle
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Load Data (Update path if running on Kaggle, e.g., '../input/german-credit-data/german_credit_data.csv')
# Here we assume the file is in the same directory for local testing
try:
    df = pd.read_csv('German_Credit_Data.csv')
except FileNotFoundError:
    print("Please upload 'German_Credit_Data.csv' or adjust the Kaggle input path.")

# Feature Engineering
df['credit_to_age'] = df['credit_amount'] / df['age']
df['payment_burden'] = df['credit_amount'] / df['duration']
df['duration_to_age'] = df['duration'] / df['age']

# Target Mapping
df['class'] = df['class'].map({'good': 0, 'bad': 1})

# One-Hot Encoding for Categoricals
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

X = df.drop('class', axis=1)
y = df['class'].values

# Strict 60/20/20 Split
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Train shape: {X_train_scaled.shape}")
print(f"Val shape: {X_val_scaled.shape}")
print(f"Test shape: {X_test_scaled.shape}")

In [ ]:
class CreditDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y).unsqueeze(1)
        
    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create DataLoaders
batch_size = 64
train_loader = DataLoader(CreditDataset(X_train_scaled, y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(CreditDataset(X_val_scaled, y_val), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(CreditDataset(X_test_scaled, y_test), batch_size=batch_size, shuffle=False)

In [ ]:
class CreditMLP(nn.Module):
    def __init__(self, input_dim):
        super(CreditMLP, self).__init__()
        
        # Shallow but wide network with aggressive regularization
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.6), # 60% probability of dropping a neuron!
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.5),
            
            nn.Linear(64, 1)
            # No Sigmoid here because we use BCEWithLogitsLoss
        )
        
    def forward(self, x):
        return self.network(x)

input_dim = X_train_scaled.shape[1]
model = CreditMLP(input_dim).to(device)
print(model)

In [ ]:
# Calculate class weight for BCEWithLogitsLoss
# We have ~70% class 0 and ~30% class 1.
num_pos = (y_train == 1).sum()
num_neg = (y_train == 0).sum()
pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=10, factor=0.5)

epochs = 100
best_val_loss = float('inf')

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            val_loss += loss.item()
            
    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    scheduler.step(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pt')
        
    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

In [ ]:
# Load the best model
model.load_state_dict(torch.load('best_model.pt'))
model.eval()

# Predict on Test Set
test_preds = []
test_probs = []
test_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch)
        probs = torch.sigmoid(logits)
        
        test_probs.extend(probs.cpu().numpy())
        test_targets.extend(y_batch.numpy())

test_probs = np.array(test_probs)
test_targets = np.array(test_targets)

# Custom threshold for precision >= 0.50
precisions, recalls, thresholds = precision_recall_curve(test_targets, test_probs)
best_thresh = 0.5
best_recall = 0.0

for p, r, t in zip(precisions, recalls, thresholds):
    if p >= 0.50 and r > best_recall:
        best_recall = r
        best_thresh = t

if best_recall == 0:
    # fallback to max f1
    f1s = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    best_thresh = thresholds[np.argmax(f1s)]

print(f"Selected Threshold: {best_thresh:.4f}")
final_preds = (test_probs >= best_thresh).astype(int)

print("\n--- Test Set Results ---")
print(f"ROC-AUC: {roc_auc_score(test_targets, test_probs):.4f}")
print(f"Accuracy: {accuracy_score(test_targets, final_preds):.4f}")
print(f"Precision: {precision_score(test_targets, final_preds):.4f}")
print(f"Recall: {recall_score(test_targets, final_preds):.4f}")
print(f"F1-Score: {f1_score(test_targets, final_preds):.4f}\n")
print(classification_report(test_targets, final_preds))